In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from constants.directories import tratados_dir

In [25]:
df_test = pd.read_csv(f'{tratados_dir}/vehiculos_test.csv')
df_train = pd.read_csv(f'{tratados_dir}/vehiculos_train.csv')

### Conversion de la variable categorica **marca** usando **target encoding**

In [26]:
%pip install category_encoders

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from category_encoders import TargetEncoder

Instanciamos y entrenamos TargetEncoder con los datos de entrenamiento.

In [28]:
encoder = TargetEncoder(cols=['marca'])

Procedemos con la conversion de los datos.

In [29]:
df_train['marca_te'] = encoder.fit_transform(df_train['marca'], df_train['fraude'])
df_test['marca_te'] = encoder.transform(df_test['marca'])

### Conversion de la variable categorica **transmision**

En este caso hemos identifica una relacion entre los tipos de transmision. Al menos entre **manual-semiautomatica-automático**. Con **híbrido** no tenemos claro si existe una relacion con las categorias anteriores por lo que lo trataremos como un caso aislado en la convertibilidad.

Este es el mapa de los tipos de transmision que planteamos donde **semi-automatic** es un punto medio entre **manual** y **automatico**.

In [30]:
transmission_map = {
    'Manual': 0,
    'Semi-Automatic': 0.5,
    'Automatic': 1,
    'Hybrid': 2
}

Procedemos a la conversion de los valores de la variable **transmision**

In [31]:
df_train['transmision_encoded'] = df_train['transmision'].map(transmission_map)
df_test['transmision_encoded'] = df_test['transmision'].map(transmission_map)

### Conversion de la variable  categorica **tipo_combustible**

In [32]:
print(df_train['tipo_combustible'].unique())

['Petrol' 'Diesel' 'Petrol Hybrid' 'Hybrid  Petrol/Electric'
 'Hybrid  Petrol/Electric Plug-in' 'Electric' 'Hybrid  Diesel/Electric'
 'Petrol Plug-in Hybrid' 'Diesel Hybrid' 'Hydrogen'
 'Hybrid  Diesel/Electric Plug-in' 'Bi Fuel' 'Petrol Ethanol'
 'Diesel Plug-in Hybrid']


Vemos que tenemos  vehículos que son híbiridos en cuanto al tipo de conbustible. En este caso aplicaremos un **one-hot-encoding**, pero no directamente a las valorres unicos presentados en el campo anterior, sino que los vamos a descomponer y ponerlos en una representacion binaria como se verá a continuación:

Los de combustion única deberían tener solo un 1 entre sus columnas, mientras que los dos 1.

In [33]:
def fuel_features(fuel):
    return pd.Series({
        'fuel_petrol': int('Petrol' in fuel),
        'fuel_diesel': int('Diesel' in fuel),
        'fuel_electric': int('Electric' in fuel),
        'fuel_hybrid': int('Hybrid' in fuel),
        'fuel_plugin': int('Plug-in' in fuel),
        'fuel_other': int(all(x not in fuel for x in ['Petrol', 'Diesel', 'Electric', 'Hybrid', 'Plug-in']))
    })

In [34]:
df_train = pd.concat([df_train, df_train['tipo_combustible'].apply(fuel_features)], axis=1)
df_test = pd.concat([df_test, df_test['tipo_combustible'].apply(fuel_features)], axis=1)

### Conversion de la variable **color**

In [35]:
print(df_train['color'].unique())

['Black' 'Grey' 'White' 'Blue' 'Yellow' 'Red' 'Orange' 'Silver' 'Gelb'
 'Beige' 'Green' 'Multicolour' 'Brown' 'Gold' 'Purple' 'Bronze' 'Pink'
 'Turquoise' 'Maroon' 'Burgundy' 'Magenta' 'Navy' 'Indigo']


Sabemos que los colores tienen relacion entre ellos. En este caso, utilizaremos la representacion vectorial RBG para representar la relacion entre ellos.

In [36]:
color_rgb_map = {
    'Black': (0, 0, 0),
    'Grey': (128, 128, 128),
    'White': (255, 255, 255),
    'Blue': (0, 0, 255),
    'Yellow': (255, 255, 0),
    'Red': (255, 0, 0),
    'Orange': (255, 165, 0),
    'Silver': (192, 192, 192),
    'Gelb': (255, 255, 0),  # 'Gelb' is German for yellow
    'Beige': (245, 245, 220),
    'Green': (0, 128, 0),
    'Multicolour': (170, 170, 170),  # Using a unique gray tone to represent multiple colors
    'Brown': (139, 69, 19),
    'Gold': (255, 215, 0),
    'Purple': (128, 0, 128),
    'Bronze': (205, 127, 50),
    'Pink': (255, 192, 203),
    'Turquoise': (64, 224, 208),
    'Maroon': (128, 0, 0),
    'Burgundy': (128, 0, 32),
    'Magenta': (255, 0, 255),
    'Navy': (0, 0, 128),
    'Indigo': (75, 0, 130)
}

In [37]:
def color_to_rgb(color):
    return color_rgb_map.get(color, (128, 128, 128))  # Default: grey

df_train[['color_r', 'color_g', 'color_b']] = df_train['color'].apply(lambda x: pd.Series(color_to_rgb(x)))
df_test[['color_r', 'color_g', 'color_b']] = df_test['color'].apply(lambda x: pd.Series(color_to_rgb(x)))

## Conversion de la variable para **tipo_vehiculo**

In [38]:
print(df_train['tipo_vehiculo'].value_counts())

tipo_vehiculo
Hatchback          76975
SUV                47465
Saloon             16706
MPV                16234
Coupe              12358
Estate             12264
Convertible         9636
Pickup              3668
Combi Van            376
Panel Van            258
Minibus              153
Wood                 107
Limousine             72
Car Derived Van       69
Window Van            50
Camper                17
Tipper                 1
Manual                 1
Name: count, dtype: int64


No parece haber una relacion aparente entre los tipos de modelos, pero vemos que tiene una cardinalidad de valores unicos significativamente grande, por lo que usaremos tearget encoding.

In [39]:
encoder = TargetEncoder(cols=['tipo_vehiculo'])
df_train['tipo_vehiculo_te'] = encoder.fit_transform(df_train['tipo_vehiculo'], df_train['fraude'])
df_test['tipo_vehiculo_te'] = encoder.transform(df_test['tipo_vehiculo'])

### Guardado de los datos convertidos

Procedemos a guardar estos datos

Limpiamos el dataset de las antiguas columnas.

In [40]:
df_train = df_train.drop(['marca','transmision','tipo_combustible','color','tipo_vehiculo'], axis=1 )
df_test = df_test.drop(['marca','transmision','tipo_combustible','color','tipo_vehiculo'], axis=1)

In [41]:
import os
from constants.directories import converted_dir

Antes de guardar los datos, nos aseguramos que lasnuevas columnas estén inegradas en los datasets finales.

In [42]:
print("Train set columns:", df_train.columns.tolist())
print("Test set columns:", df_test.columns.tolist())

Train set columns: ['modelo', 'anio_registro', 'millas_recorridas', 'tamanio_motor', 'precio_vehiculo', 'num_asientos', 'num_puertas', 'problema_averia', 'id_problema_averia', 'fecha_averia', 'complejidad_reparacion', 'costo_reparacion', 'horas_reparacion', 'fecha_reparacion', 'fraude', 'marca_te', 'transmision_encoded', 'fuel_petrol', 'fuel_diesel', 'fuel_electric', 'fuel_hybrid', 'fuel_plugin', 'fuel_other', 'color_r', 'color_g', 'color_b', 'tipo_vehiculo_te']
Test set columns: ['index', 'modelo', 'anio_registro', 'millas_recorridas', 'tamanio_motor', 'precio_vehiculo', 'num_asientos', 'num_puertas', 'problema_averia', 'id_problema_averia', 'fecha_averia', 'complejidad_reparacion', 'costo_reparacion', 'horas_reparacion', 'fecha_reparacion', 'marca_te', 'transmision_encoded', 'fuel_petrol', 'fuel_diesel', 'fuel_electric', 'fuel_hybrid', 'fuel_plugin', 'fuel_other', 'color_r', 'color_g', 'color_b', 'tipo_vehiculo_te']


Adicioanalmente que las columnas del dataset de entrenameinto tambien eesten en el de testeo.

In [43]:
for col in df_train.columns:
    if col not in df_test.columns:
        df_test[col] = 0

# Verify the shapes before saving
print("\nTrain set shape:", df_train.shape)
print("Test set shape:", df_test.shape)


Train set shape: (196410, 27)
Test set shape: (21822, 28)


procedemos a guardar los datos en un directorio

In [44]:
os.makedirs(converted_dir, exist_ok=True)

In [45]:
df_train.to_csv(f"{converted_dir}/vehiculos_train.csv", index=False)
df_test.to_csv(f"{converted_dir}/vehiculos_test.csv", index=False)